# Kubeflow Pipeline

Develop the YOLO training pipeline, one step at a time.

```txt
fetch_data -> prepare_data -> train -> evaluate -> upload_model
```


## Environment

In [1]:
# pip install
%pip install -q -U kfp

Note: you may need to restart the kernel to use updated packages.


In [2]:
from kfp import compiler, dsl
from kfp.dsl import Dataset, Input, Output


---

## Step 1: `fetch_data`

Pull the DVC-tracked raw dataset out of S3.


In [3]:
@dsl.component(base_image="python:3.12", packages_to_install=["boto3"])
def fetch_data(
    bucket: str,
    dvc_dir_hash: str,
    region: str,
    raw: Output[Dataset],
):
    """Restore the DVC-tracked raw dataset from S3 into the `raw` artifact."""
    import json
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    import boto3

    s3 = boto3.client("s3", region_name=region)

    def dvc_key(md5: str) -> str:
        # DVC shards its content-addressed store by the first two hex chars.
        return "dvcstore/files/md5/" + md5[:2] + "/" + md5[2:]

    # the .dir object lists {"md5": ..., "relpath": ...} for every file
    manifest = json.loads(
        s3.get_object(Bucket=bucket, Key=dvc_key(dvc_dir_hash))["Body"].read()
    )

    root = Path(raw.path)
    root.mkdir(parents=True, exist_ok=True)

    def fetch(entry):
        target = root / entry["relpath"]
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, dvc_key(entry["md5"]), str(target))

    # 1100+ small objects: this is latency-bound, not bandwidth-bound
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(fetch, manifest))

    images = [p for p in root.iterdir() if p.suffix.lower() in {".jpeg", ".jpg", ".png"}]
    if not images:
        raise RuntimeError("no images restored -- check the dvc_dir_hash")

    # metadata shows up on the artifact in the run UI
    raw.metadata["files"] = len(manifest)
    raw.metadata["images"] = len(images)
    raw.metadata["dvc_dir_hash"] = dvc_dir_hash

    print("fetched", len(manifest), "files /", len(images), "images to", root)

---

## Step 2: `prepare_data`

Split into train/val and write the ultralytics descriptor.


In [4]:
@dsl.component(base_image="python:3.12", packages_to_install=["pyyaml"])
def prepare_data(
    raw: Input[Dataset],
    val_fraction: float,
    split_seed: int,
    processed: Output[Dataset],
):
    """Build processed/{train,val}/{images,labels} plus data.yaml."""
    import random
    import shutil
    from pathlib import Path

    import yaml

    suffixes = {".jpeg", ".jpg", ".png"}
    src = Path(raw.path)
    dst = Path(processed.path)

    # images and labels pair by basename: foo.jpeg <-> foo.txt
    stems = sorted(p.stem for p in src.iterdir() if p.suffix.lower() in suffixes)
    random.Random(split_seed).shuffle(stems)
    cut = int(len(stems) * (1 - val_fraction))

    counts = {}
    for split, names in (("train", stems[:cut]), ("val", stems[cut:])):
        for sub in ("images", "labels"):
            (dst / split / sub).mkdir(parents=True, exist_ok=True)
        for stem in names:
            image = next(p for p in src.glob(stem + ".*") if p.suffix.lower() in suffixes)
            shutil.copy(image, dst / split / "images" / image.name)
            label = src / (stem + ".txt")
            # an image with no label file is a legitimate negative sample
            if label.exists():
                shutil.copy(label, dst / split / "labels" / label.name)
        counts[split] = len(names)

    if not counts["train"] or not counts["val"]:
        raise RuntimeError("empty split: " + str(counts))

    class_names = (src / "classes.txt").read_text().split()
    (dst / "data.yaml").write_text(
        yaml.safe_dump(
            {
                "path": str(dst),          # absolute, see above
                "train": "train/images",
                "val": "val/images",
                "nc": len(class_names),
                "names": class_names,
            },
            sort_keys=False,
        )
    )

    processed.metadata.update(counts)
    processed.metadata["classes"] = class_names
    print("split", counts, "classes", class_names)
    print((dst / "data.yaml").read_text())

---

## Pipeline

`raw=fetch.outputs["raw"]` creates the dependency; KFP derives the DAG from the
data flow.

In [5]:
@dsl.pipeline(name="yolo-data")
def data_pipeline(
    bucket: str = "kubeflow-yolo-dev-099139718958",
    dvc_dir_hash: str = "0e94102a7a6b4424a0f1292c2f221072.dir",
    region: str = "ca-central-1",
    val_fraction: float = 0.2,
    split_seed: int = 0,
):
    fetch = fetch_data(
        bucket=bucket,
        dvc_dir_hash=dvc_dir_hash,
        region=region,
    ).set_caching_options(True)

    prepare_data(
        raw=fetch.outputs["raw"],
        val_fraction=val_fraction,
        split_seed=split_seed,
    )

Compile. Needs no cluster: this checks the components and the DAG.

In [6]:
PACKAGE = "data_pipeline.yaml"

# compile pipeline
compiler.Compiler().compile(data_pipeline, PACKAGE)

print("compiled", PACKAGE)

compiled data_pipeline.yaml


---

## Submit

Run it. This is the only part tied to the notebook: `ml-pipeline-ui` is a
ClusterIP service, and the token comes from the `kfp-api-token` PodDefault.
Elsewhere the compiled yaml is submitted however that system authenticates.

In [ ]:
import kfp
from kfp.client.set_volume_credentials import ServiceAccountTokenVolumeCredentials

NAMESPACE = "kubeflow-user-example-com"
HOST = "http://ml-pipeline-ui.kubeflow.svc.cluster.local"
TOKEN_PATH = "/var/run/secrets/kubeflow/pipelines/token"

client = kfp.Client(
    host=HOST,
    credentials=ServiceAccountTokenVolumeCredentials(path=TOKEN_PATH),
)

# reuses the experiment if it already exists
experiment = client.create_experiment(name="yolo-data", namespace=NAMESPACE)

# submit run
run = client.run_pipeline(
    experiment_id=experiment.experiment_id,
    job_name="data-split",
    pipeline_package_path=PACKAGE,
)

print("run", run.run_id)

/opt/conda/lib/python3.12/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(
ERROR:root:Failed to get healthz info attempt 1 of 5.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/kfp/client/client.py", line 424, in get_kfp_healthz
    return self._healthz_api.healthz_service_get_healthz()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/kfp_server_api/api/healthz_service_api.py", line 63, in healthz_service_get_healthz
    return self.healthz_service_get_healthz_with_http_info(**kwargs)  # noqa: E501
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/kfp_server_api/api/healthz_service_api.py", line 134, in healthz_service_get_healthz_with_http_info
    return self.api_client.call_api(
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/

In [ ]:
# wait for run
result = client.wait_for_run_completion(run.run_id, timeout=1800)

print("state", result.state)

---

## Next

`train` - fine-tune `yolo11n.pt` on `processed`.